The goal of this notebook is to quantify how the plankton inside a Gulf Stream eddy changes over its lifetime, from total chlorophyll through the plankton groups to the pigments, for cyclones traveling south and anticyclones traveling north. The three sources are:

- Copernicus CHL, the daily 4 km multi-sensor product in NASA 8-day composites, the only CHL product of the analysis.
- Copernicus plankton groups, the chlorophyll-a attributed to each phytoplankton group by the same product: five functional types (diatoms, dinophytes, haptophytes, green algae, prochlorophytes), prokaryotes, and the micro and pico size classes. The nano size class equals the haptophytes on every row, so it is left out.
- SDP pigments, the 13 concentrations that the Kramer et al. (2022) Spectral Derivative Pigments model retrieves from the 8-day PACE Rrs composite inside each eddy: total chlorophyll-a and 12 accessory pigments. PACE begins in March 2024, so this record is shorter and fewer target eddies have a composite. SDP clips a negative prediction to zero, and those pixels stay in the means.

Target eddies:
- Cyclones formed north of the Gulf Stream axis and ended south, or formed within `NEAR_AXIS_KM` (150 km) of the axis and ended south. Anticyclones use the reversed rule.

Eddy requirements:
- Copernicus: 50% CHL coverage of the speed-contour interior and at least 10 valid pixels per eddy-composite. A group keeps the same rule on its own pixels, because the group mask is smaller than the CHL mask.
- SDP: 80% Rrs coverage of the interior and at least 10 valid pixels, from `collocate_pace` and `build_gold_table`.
- Within each age bin, all pixels within each eddy-composite are averaged, then the composites of each eddy, and the eddies of each polarity are averaged last with a bootstrap over eddies for the 95% interval.

Views, repeated for each source:
- Over lifetime: the interior mean per age bin and its change from the first observation.
- Age and radius: the mean in each age bin and each 0.2 R ring from the eddy center out to 2 radii, R being the speed radius. A ring needs 3 valid pixels, and a cell needs 3 eddies.

The last figure puts every variable on one axis: the change by the last fifth of the track as a percent of the first observation.

The ring step writes its table to `gold/eddy_plankton_rings.parquet` on the first run (the ring means of CHL and every group for every eddy-composite in the plankton table) and reads it after that. Delete the file to recompute it.

In [ ]:
from pathlib import Path
from typing import cast
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.colorbar import Colorbar
from matplotlib.colors import Normalize
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.config import load_config
from eddy_tracking.packages.py_eddy_tracker.observations.tracking import TrackEddiesObservations
from eddy_tracking.preprocess.tracks import PET_EPOCH

EXPERIMENT = 'gulf_stream_20240305_20260531'
N_AGE_BINS = 5
N_RADIAL_BINS = 10
MAX_RADIUS = 2
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
EXCLUDE_RECORD_EDGE_TRACKS = False
NEAR_AXIS_KM = 150
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_config(EXPERIMENT)
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
target_labels = {'cyclone': 'Target cyclones', 'anticyclone': 'Target anticyclones'}
identity_columns = ['polarity', 'track_id']
groups = ['DIATO', 'DINO', 'HAPTO', 'GREEN', 'PROCHLO', 'MICRO', 'PICO', 'PROKAR']
group_labels = {
    'DIATO': 'Diatoms', 'DINO': 'Dinophytes', 'HAPTO': 'Haptophytes', 'GREEN': 'Green algae',
    'PROCHLO': 'Prochlorophytes', 'MICRO': 'Microphytoplankton', 'PICO': 'Picophytoplankton', 'PROKAR': 'Prokaryotes',
}
pigments = ['Tchla', 'Fuco', 'HexFuco', 'ButFuco', 'Perid', 'Chlc12', 'Chlc3', 'Allo', 'MV_chlb', 'Neo', 'Viola', 'Zea', 'DV_chla']
pigment_labels = {
    'Tchla': 'Total chlorophyll-a', 'Fuco': 'Fucoxanthin', 'HexFuco': "19'-Hex-fucoxanthin", 'ButFuco': "19'-But-fucoxanthin",
    'Perid': 'Peridinin', 'Chlc12': 'Chlorophyll-c1+c2', 'Chlc3': 'Chlorophyll-c3', 'Allo': 'Alloxanthin',
    'MV_chlb': 'Monovinyl chlorophyll-b', 'Neo': 'Neoxanthin', 'Viola': 'Violaxanthin',
    'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a',
}
variable_labels = {'CHL': 'CHL', **group_labels, **pigment_labels}
panel_letters = 'abcdefghijklmnop'
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
radial_edges = np.linspace(0, MAX_RADIUS, N_RADIAL_BINS + 1)

eddy_tracks = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
plankton = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet')
pigment_table = pd.read_parquet(DATA_DIR / 'gold/eddy_pigment_table.parquet')
pigment_table['polarity'] = cast(pd.Series, pigment_table['polarity']).map({0: 'anticyclone', 1: 'cyclone'})
pigment_table = pigment_table.rename(columns={f'eddy_mean_{pigment}': pigment for pigment in pigments})
physical_start, physical_end = pd.to_datetime(cfg['base']['time']['eddy_date_range'])
eddy_tracks = eddy_tracks.merge(
    cast(pd.Series, plankton.groupby(identity_columns).size()).rename('n_chl_dates').reset_index(),
    on=identity_columns, how='left',
)
eddy_tracks['n_chl_dates'] = eddy_tracks['n_chl_dates'].fillna(0).astype(int)
eddy_tracks['at_record_edge'] = (
    (eddy_tracks['birth_date'] <= physical_start)
    | (eddy_tracks['death_date'] >= physical_end)
)
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['crossed_axis'] = eddy_tracks['movement'].eq(target_class)
eddy_tracks['near_axis_birth'] = (
    eddy_tracks['birth_distance_km'].abs().le(NEAR_AXIS_KM)
    & eddy_tracks['death_side'].eq(target_class.str[1])
)
eddy_tracks['is_target'] = eddy_tracks['crossed_axis'] | eddy_tracks['near_axis_birth']
targets = eddy_tracks.loc[eddy_tracks['is_target'] & ~(EXCLUDE_RECORD_EDGE_TRACKS & eddy_tracks['at_record_edge']), identity_columns]
target_plankton = plankton.merge(targets, on=identity_columns)
target_pigments = pigment_table.merge(targets, on=identity_columns)
plankton_settings = cfg['collocate_plankton']
for group in groups:
    covered = (
        target_plankton[f'{group}_n_pixels'].ge(plankton_settings['min_pixels'])
        & target_plankton[f'{group}_n_pixels'].div(target_plankton['n_pixels']).ge(plankton_settings['min_coverage'])
    )
    target_plankton.loc[~covered, group] = np.nan
analysis = pd.concat([
    target_plankton[identity_columns + ['date', 'age_frac']].assign(source='chl', variable='CHL', concentration=target_plankton['CHL']),
    target_plankton[identity_columns + ['date', 'age_frac'] + groups].melt(id_vars=identity_columns + ['date', 'age_frac'], var_name='variable', value_name='concentration').assign(source='plankton'),
    target_pigments[identity_columns + ['date', 'age_frac'] + pigments].melt(id_vars=identity_columns + ['date', 'age_frac'], var_name='variable', value_name='concentration').assign(source='sdp'),
], ignore_index=True).dropna(subset=['concentration']).sort_values(['source', 'variable'] + identity_columns + ['date'], ignore_index=True)

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

display(cast(pd.DataFrame, analysis.drop_duplicates(['source'] + identity_columns + ['date']).groupby(['source', 'polarity']).agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'),
    first_composite=('date', 'min'), last_composite=('date', 'max'),
)))

In [ ]:
classes = ['NN', 'NS', 'SN', 'SS']
bar_positions = np.arange(len(classes))
count_rows = []
matched_tracks = eddy_tracks.loc[eddy_tracks['n_chl_dates'].gt(0)]
fig, ax = cast(tuple[Figure, Axes], plt.subplots(figsize=(3.35, 2.7), layout='constrained'))
for offset, polarity in ((-0.2, 'cyclone'), (0.2, 'anticyclone')):
    tracks = matched_tracks.loc[matched_tracks['polarity'].eq(polarity)]
    total = tracks.groupby('movement').size().reindex(classes, fill_value=0).to_numpy()
    target = tracks.loc[tracks['is_target']].groupby('movement').size().reindex(classes, fill_value=0).to_numpy()
    ax.bar(bar_positions + offset, total, width=0.36, facecolor='white', edgecolor=polarity_colors[polarity], linewidth=0.8)
    ax.bar(bar_positions + offset, target, width=0.36, facecolor=polarity_colors[polarity], edgecolor=polarity_colors[polarity], linewidth=0.8)
    for position, n_total, n_target in zip(bar_positions + offset, total, target):
        ax.text(position, n_total + 8, f'{n_target} of {n_total}' if n_target else str(n_total), ha='center', va='bottom', rotation=90)
    for label, n_total, n_target in zip(classes, total, target):
        count_rows.append({'polarity': polarity, 'class': label, 'tracks': int(n_total), 'target': int(n_target)})
ax.xaxis.set_ticks(bar_positions, [label[0] + '→' + label[1] for label in classes])
ax.tick_params(axis='x', length=0)
ax.set_ylim(0, 400)
ax.set_xlabel('First → last side of axis')
ax.set_ylabel('Tracks')
ax.set_title('Lagrangian eddy track counts', loc='left')
ax.legend(handles=[
    Patch(facecolor='white', edgecolor='#444444', linewidth=0.8, label='All tracks'),
    Patch(facecolor='#444444', edgecolor='#444444', linewidth=0.8, label='Target eddies'),
], loc='upper left', handlelength=1.2, handleheight=0.9, borderaxespad=0.2)
class_counts = pd.DataFrame(count_rows)
plt.show()
display(class_counts)
target_rules = {
    'crossed the axis': eddy_tracks['crossed_axis'],
    f'born within {NEAR_AXIS_KM} km, no crossing': eddy_tracks['near_axis_birth'] & ~eddy_tracks['crossed_axis'],
    'target eddies': eddy_tracks['is_target'],
}
display(pd.DataFrame([
    {'polarity': polarity, 'rule': rule, 'tracks': int((members & eddy_tracks['polarity'].eq(polarity)).sum())}
    for polarity in polarity_names for rule, members in target_rules.items()
]))

In [ ]:
analysis['age_bin'] = np.minimum((analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1)
analysis['initial'] = analysis.groupby(['source', 'variable'] + identity_columns)['concentration'].transform('first')
analysis['change'] = analysis['concentration'] - analysis['initial']
eddy_bins = cast(pd.DataFrame, analysis.groupby(['source', 'variable'] + identity_columns + ['age_bin']).agg(
    concentration=('concentration', 'mean'), change=('change', 'mean'), initial=('initial', 'first'),
)).reset_index()
n_eddies = analysis.groupby(['source', 'polarity'])['track_id'].nunique()
rng = np.random.default_rng(RANDOM_SEED)
summary_rows = []
for keys, source_bins in eddy_bins.groupby(['source', 'polarity']):
    source, polarity = cast(tuple[str, str], keys)
    eddy_ids = sorted(source_bins['track_id'].unique())
    draws = rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))
    for variable, variable_bins in source_bins.groupby('variable'):
        initial = variable_bins.groupby('track_id')['initial'].first().reindex(eddy_ids).to_numpy(dtype=float)
        for metric in ('concentration', 'change', 'percent'):
            matrix = variable_bins.pivot(index='track_id', columns='age_bin', values='change' if metric == 'percent' else metric).reindex(index=eddy_ids, columns=range(N_AGE_BINS)).to_numpy(dtype=float)
            present = np.isfinite(matrix)
            counts = present.sum(axis=0)
            denominator = np.where(present, initial[:, None], 0).sum(axis=0) / 100 if metric == 'percent' else counts
            means = np.divide(np.nansum(matrix, axis=0), denominator, out=np.full(N_AGE_BINS, np.nan), where=counts > 0)
            sampled = matrix[draws]  # (n_eddies, n_bins) -> (n_bootstrap, n_eddies, n_bins)
            sampled_present = np.isfinite(sampled)
            sampled_counts = sampled_present.sum(axis=1)
            sampled_denominator = np.where(sampled_present, initial[draws][:, :, None], 0).sum(axis=1) / 100 if metric == 'percent' else sampled_counts
            sampled_means = np.divide(np.nansum(sampled, axis=1), sampled_denominator, out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0)
            low, high = np.nanquantile(sampled_means, [0.025, 0.975], axis=0)
            summary_rows.append(pd.DataFrame({
                'source': source, 'polarity': polarity, 'variable': variable, 'metric': metric, 'age_bin': range(N_AGE_BINS),
                'age_midpoint': bin_centers, 'mean': means, 'ci_low': np.where(counts >= 3, low, np.nan), 'ci_high': np.where(counts >= 3, high, np.nan), 'n_eddies': counts,
            }))
lifetime_summary = pd.concat(summary_rows, ignore_index=True)
heat_cmap = plt.get_cmap('viridis').copy()
heat_cmap.set_bad('#e6e6e6')


def draw_lifetime(ax: Axes, source: str, variable: str, metric: str) -> None:
    if metric != 'concentration':
        ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
    for polarity, color in polarity_colors.items():
        result = lifetime_summary.loc[
            lifetime_summary['source'].eq(source) & lifetime_summary['variable'].eq(variable)
            & lifetime_summary['polarity'].eq(polarity) & lifetime_summary['metric'].eq(metric)
        ].sort_values('age_bin')
        intervals = result['ci_low'].notna()
        ax.errorbar(
            result.loc[intervals, 'age_midpoint'], result.loc[intervals, 'mean'],
            yerr=[result.loc[intervals, 'mean'] - result.loc[intervals, 'ci_low'], result.loc[intervals, 'ci_high'] - result.loc[intervals, 'mean']],
            fmt='none', ecolor=color, capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=3,
        )
        ax.plot(result['age_midpoint'], result['mean'], '-o', color=color, linewidth=1.2, markersize=3.2, markeredgecolor='white', markeredgewidth=0.5, label=f'{target_labels[polarity]} (n = {n_eddies[source, polarity]})', zorder=4)
    ax.xaxis.set_ticks(np.linspace(0, 1, 6))
    ax.set_xlim(0, 1)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
    ax.yaxis.set_major_locator(locator)
    ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
    ax.yaxis.set_ticks(ticks)
    ax.set_ylim(ticks[0], ticks[-1])


def summarize_cells(rings: pd.DataFrame, fields: list[str]) -> pd.DataFrame:
    radial = rings.melt(id_vars=identity_columns + ['date', 'age_frac', 'radial_bin'], value_vars=fields, var_name='variable', value_name='concentration')
    radial['age_bin'] = np.minimum((radial['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1)
    cells = cast(pd.DataFrame, radial.groupby(identity_columns + ['variable', 'age_bin', 'radial_bin'])['concentration'].mean().reset_index().groupby(['polarity', 'variable', 'age_bin', 'radial_bin']).agg(
        concentration=('concentration', 'mean'), n_eddies=('concentration', 'count'),
    )).reset_index()
    cells.loc[cells['n_eddies'].lt(3), 'concentration'] = np.nan
    return cells


def draw_age_radius(fig: Figure, axes: np.ndarray, cells: pd.DataFrame) -> Colorbar:
    norm = Normalize(cells['concentration'].min(), cells['concentration'].max())
    for ax, polarity in zip(axes, polarity_names):
        ax = cast(Axes, ax)
        grid = cells.loc[cells['polarity'].eq(polarity)].pivot(index='radial_bin', columns='age_bin', values='concentration').reindex(index=range(N_RADIAL_BINS), columns=range(N_AGE_BINS)).to_numpy(dtype=float)
        ax.pcolormesh(bin_edges, radial_edges, np.ma.masked_invalid(grid), cmap=heat_cmap, norm=norm, edgecolors='white', linewidth=0.5)
        ax.axhline(1, color='#222222', linewidth=0.8, linestyle=(0, (4, 2.5)), zorder=3)
        ax.set_aspect('equal')
        ax.set_title(f'{polarity.capitalize()}s', pad=2)
        ax.xaxis.set_ticks([0, 0.5, 1], ['0', '0.5', '1'])
        ax.xaxis.set_ticks(bin_edges, minor=True)
        ax.yaxis.set_ticks([0, 0.5, 1, 1.5, 2], ['0', '0.5', '1', '1.5', '2'])
        ax.yaxis.set_ticks(radial_edges, minor=True)
        ax.tick_params(length=2)
        ax.tick_params(which='minor', length=1.2)
    colorbar = fig.colorbar(ScalarMappable(norm=norm, cmap=heat_cmap), cax=cast(Axes, axes[1]).inset_axes((1.12, 0, 0.1, 1)))
    colorbar.ax.tick_params(length=2)
    colorbar.ax.yaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10]))
    colorbar.outline.set_linewidth(0.5)
    return colorbar


display(lifetime_summary.loc[lifetime_summary['metric'].eq('concentration')].groupby(['source', 'polarity', 'age_bin'])['n_eddies'].min().unstack('age_bin'))

## Copernicus CHL

In [ ]:
plankton_fields = ['CHL', 'DIATO', 'DINO', 'GREEN', 'HAPTO', 'MICRO', 'NANO', 'PICO', 'PROCHLO', 'PROKAR']
plankton_rings_path = DATA_DIR / 'gold/eddy_plankton_rings.parquet'
if not plankton_rings_path.exists():
    track_observations = []
    for polarity in polarity_names:
        tracked = TrackEddiesObservations.load_file(str(DATA_DIR / f'silver/eddy_track/{polarity}/{polarity}_tracks.zarr'))
        keep = ~tracked.virtual.astype(bool)
        track_observations.append(pd.DataFrame({
            'polarity': polarity, 'track_id': tracked.track[keep].astype(int),
            'day': pd.Timestamp(PET_EPOCH) + pd.to_timedelta(tracked.time[keep].astype(int), unit='D'),
            'center_lon': (tracked.longitude[keep] + 180) % 360 - 180, 'center_lat': tracked.latitude[keep],
            'radius_km': tracked.radius_s[keep] / 1000,
        }))
    track_observations = pd.concat(track_observations, ignore_index=True)
    date_ranges = []
    for year in range(physical_start.year, physical_end.year + 1):
        start = pd.Timestamp(year, 1, 1)
        while start.year == year:
            end = min(start + pd.Timedelta(days=7), pd.Timestamp(year, 12, 31))
            if end >= physical_start and start <= physical_end:
                date_ranges.append((start, end))
            start = end + pd.Timedelta(days=1)
    fields = xr.open_mfdataset(sorted((DATA_DIR / 'bronze/plankton').glob('plankton_*.nc')), combine='by_coords')[plankton_fields]
    lon = fields['longitude'].to_numpy()
    lat = fields['latitude'].to_numpy()
    rings = []
    for start, end in date_ranges:
        composites = plankton.loc[plankton['date'].between(start, end), identity_columns + ['date']]
        if composites.empty:
            continue
        candidates = track_observations.loc[track_observations['day'].between(start, end)].merge(composites, on=identity_columns)
        candidates['offset'] = (candidates['day'] - candidates['date']).abs()
        composite = fields.sel(time=slice(start, end)).mean('time').load()
        for eddy in candidates.sort_values(['offset', 'day']).drop_duplicates(identity_columns).itertuples():
            half_width = MAX_RADIUS * eddy.radius_km / 111.32
            lon_index = np.flatnonzero(np.abs(lon - eddy.center_lon) <= half_width / np.cos(np.radians(eddy.center_lat)))
            lat_index = np.flatnonzero(np.abs(lat - eddy.center_lat) <= half_width)
            lon_grid, lat_grid = np.meshgrid(lon[lon_index], lat[lat_index])
            half_chord = (
                np.sin(np.radians(lat_grid - eddy.center_lat) / 2) ** 2
                + np.cos(np.radians(lat_grid)) * np.cos(np.radians(eddy.center_lat)) * np.sin(np.radians(lon_grid - eddy.center_lon) / 2) ** 2
            )
            distance_km = 2 * 6371 * np.arcsin(np.sqrt(half_chord))
            radial_bin = np.digitize(distance_km.ravel() / eddy.radius_km, radial_edges) - 1
            inside = radial_bin < N_RADIAL_BINS
            n_pixels = np.bincount(radial_bin[inside], minlength=N_RADIAL_BINS)
            ring = {
                'polarity': eddy.polarity, 'track_id': eddy.track_id, 'date': eddy.date,
                'radial_bin': np.arange(N_RADIAL_BINS), 'n_pixels': n_pixels,
            }
            box = composite.isel(longitude=lon_index, latitude=lat_index)
            for field in plankton_fields:
                values = box[field].to_numpy().ravel()
                valid = inside & np.isfinite(values)
                n_valid = np.bincount(radial_bin[valid], minlength=N_RADIAL_BINS)
                covered = (n_valid >= 3) & (n_valid >= plankton_settings['min_coverage'] * n_pixels)
                ring[field] = np.where(covered, np.bincount(radial_bin[valid], weights=values[valid], minlength=N_RADIAL_BINS) / np.maximum(n_valid, 1), np.nan)
                ring[f'{field}_n_pixels'] = n_valid
            rings.append(pd.DataFrame(ring))
    pd.concat(rings, ignore_index=True).to_parquet(plankton_rings_path, index=False)
plankton_cells = summarize_cells(pd.read_parquet(plankton_rings_path).merge(target_plankton[identity_columns + ['date', 'age_frac']], on=identity_columns + ['date']), ['CHL'] + groups)

chl_cells = plankton_cells.loc[plankton_cells['variable'].eq('CHL')]

In [ ]:
chl_fig = cast(Figure, plt.figure(figsize=(6.69, 2.8)))
outer = chl_fig.add_gridspec(1, 3, left=0.085, right=0.9, bottom=0.25, top=0.88, width_ratios=(1.5, 1.5, 1.7), wspace=0.4)
line_axes = [cast(Axes, chl_fig.add_subplot(outer[index])) for index in range(2)]
heat_axes = cast(np.ndarray, outer[2].subgridspec(1, 2, wspace=0.2).subplots(sharey=True))
for ax, letter, metric, title, ylabel in zip(
    line_axes, 'ab', ('concentration', 'change'), ('Interior CHL', 'Change in CHL'), ('CHL (mg m$^{-3}$)', '$\\Delta$CHL (mg m$^{-3}$)'),
):
    draw_lifetime(ax, 'chl', 'CHL', metric)
    ax.set_title(f'$\\bf{{({letter})}}$ {title}', loc='left', pad=12.4)
    ax.set_xlabel('Fraction of observed track')
    ax.set_ylabel(ylabel)
draw_age_radius(chl_fig, heat_axes, chl_cells).set_label('CHL (mg m$^{-3}$)')
cast(Axes, heat_axes[0]).set_ylabel('Distance from center (speed radii)')
cast(Axes, heat_axes[0]).text(0, 1.1, '$\\bf{(c)}$ CHL by radius and age', transform=heat_axes[0].transAxes, va='bottom', ha='left')
cast(Axes, heat_axes[0]).set_xlabel('Fraction of observed track')
cast(Axes, heat_axes[0]).xaxis.set_label_coords(1.1, -0.12)
chl_fig.legend(*line_axes[0].get_legend_handles_labels(), loc='lower center', ncol=2, handlelength=2.2)
plt.show()
display(lifetime_summary.loc[lifetime_summary['source'].eq('chl')].drop(columns=['source', 'variable']).round(4))
display(chl_cells.pivot(index=['polarity', 'age_bin'], columns='radial_bin', values='concentration').round(3))

## Copernicus plankton groups

In [ ]:
for metric, ylabel in (
    ('concentration', 'Group chlorophyll-a (mg m$^{-3}$)'),
    ('change', 'Change from first observation (mg m$^{-3}$)'),
    ('percent', 'Change from first observation (%)'),
):
    life_fig, life_axes = cast(tuple[Figure, np.ndarray], plt.subplots(3, 3, figsize=(6.69, 6.1), sharex=True, layout='constrained'))
    for letter, ax, group in zip(panel_letters, life_axes.flat, groups):
        ax = cast(Axes, ax)
        draw_lifetime(ax, 'plankton', group, metric)
        ax.set_title(f'$\\bf{{({letter})}}$ {group_labels[group]}', loc='left')
    for ax in (life_axes[-1, 0], life_axes[-1, 1], life_axes[-2, 2]):
        ax = cast(Axes, ax)
        ax.tick_params(labelbottom=True)
        ax.set_xlabel('Fraction of observed track')
    legend_ax = cast(Axes, life_axes.flat[len(groups)])
    legend_ax.axis('off')
    legend_ax.legend(*cast(Axes, life_axes.flat[0]).get_legend_handles_labels(), loc='upper left', handlelength=1.6).set_in_layout(False)
    life_fig.supylabel(ylabel, fontsize=8)
    plt.show()
display(lifetime_summary.loc[lifetime_summary['source'].eq('plankton') & lifetime_summary['metric'].eq('concentration')].pivot(index='variable', columns=['polarity', 'age_bin'], values='mean').reindex(groups).round(4))

In [ ]:
radial_fig = cast(Figure, plt.figure(figsize=(6.69, 5.4)))
outer = radial_fig.add_gridspec(3, 3, left=0.075, right=0.92, bottom=0.075, top=0.935, wspace=0.6, hspace=0.6)
for index, (letter, group) in enumerate(zip(panel_letters, groups)):
    axes = cast(np.ndarray, outer[index].subgridspec(1, 2, wspace=0.12).subplots(sharey=True))
    draw_age_radius(radial_fig, axes, plankton_cells.loc[plankton_cells['variable'].eq(group)])
    cast(Axes, axes[0]).text(0, 1.14, f'$\\bf{{({letter})}}$ {group_labels[group]}', transform=axes[0].transAxes, va='bottom', ha='left')
radial_fig.supxlabel('Fraction of observed track', fontsize=8)
radial_fig.supylabel('Distance from eddy center (speed radii)', fontsize=8)
plt.show()
display(plankton_cells.groupby(['radial_bin', 'polarity', 'age_bin'])['n_eddies'].min().unstack(['polarity', 'age_bin']))

## SDP pigments

In [ ]:
for metric, ylabel in (
    ('concentration', 'Pigment concentration (mg m$^{-3}$)'),
    ('change', 'Change from first observation (mg m$^{-3}$)'),
    ('percent', 'Change from first observation (%)'),
):
    life_fig, life_axes = cast(tuple[Figure, np.ndarray], plt.subplots(5, 3, figsize=(6.69, 7.4), sharex=True, layout='constrained'))
    cast(Axes, life_axes.flat[len(pigments) + 1]).remove()
    for letter, ax, pigment in zip(panel_letters, life_axes.flat, pigments):
        ax = cast(Axes, ax)
        draw_lifetime(ax, 'sdp', pigment, metric)
        ax.set_title(f'$\\bf{{({letter})}}$ {pigment_labels[pigment]}', loc='left')
    for ax in (life_axes[-1, 0], *life_axes[-2, 1:]):
        ax = cast(Axes, ax)
        ax.tick_params(labelbottom=True)
        ax.set_xlabel('Fraction of observed track')
    legend_ax = cast(Axes, life_axes.flat[len(pigments)])
    legend_ax.axis('off')
    legend_ax.legend(*cast(Axes, life_axes.flat[0]).get_legend_handles_labels(), loc='upper left').set_in_layout(False)
    life_fig.supylabel(ylabel, fontsize=8)
    plt.show()
display(lifetime_summary.loc[lifetime_summary['source'].eq('sdp') & lifetime_summary['metric'].eq('concentration')].pivot(index='variable', columns=['polarity', 'age_bin'], values='mean').reindex(pigments).round(4))

In [ ]:
pixel_columns = {'T chla': 'Tchla', 'DV chla': 'DV_chla', 'MV chlb': 'MV_chlb', 'chl c1+c2': 'Chlc12', 'chl c3': 'Chlc3'}
pixels = pd.concat([
    pd.read_parquet(DATA_DIR / f'silver/pigments/{polarity}/eddy_{track_id}_pigments.parquet').assign(polarity=polarity)
    for polarity, track_id in target_pigments[identity_columns].drop_duplicates().itertuples(index=False)
], ignore_index=True).rename(columns=pixel_columns)
pixels = pixels.merge(target_pigments[identity_columns + ['date', 'age_frac']], on=identity_columns + ['date'])
half_chord = (
    np.sin(np.radians(pixels['pixel_lat'] - pixels['center_lat']) / 2) ** 2
    + np.cos(np.radians(pixels['pixel_lat'])) * np.cos(np.radians(pixels['center_lat'])) * np.sin(np.radians(pixels['pixel_lon'] - pixels['center_lon']) / 2) ** 2
)
pixels['radial_bin'] = np.digitize(2 * 6371 * np.arcsin(np.sqrt(half_chord)) / pixels['radius_km'], radial_edges) - 1
rings = cast(pd.DataFrame, pixels.loc[pixels['radial_bin'].lt(N_RADIAL_BINS)].groupby(identity_columns + ['date', 'age_frac', 'radial_bin']).agg(
    n_pixels=('Tchla', 'size'), **{pigment: (pigment, 'mean') for pigment in pigments},
)).reset_index()
rings.loc[rings['n_pixels'].lt(3), pigments] = np.nan
pigment_cells = summarize_cells(rings, pigments)

radial_fig = cast(Figure, plt.figure(figsize=(6.69, 8.9)))
outer = radial_fig.add_gridspec(5, 3, left=0.075, right=0.92, bottom=0.045, top=0.96, wspace=0.6, hspace=0.6)
for index, (letter, pigment) in enumerate(zip(panel_letters, pigments)):
    axes = cast(np.ndarray, outer[index].subgridspec(1, 2, wspace=0.12).subplots(sharey=True))
    draw_age_radius(radial_fig, axes, pigment_cells.loc[pigment_cells['variable'].eq(pigment)])
    cast(Axes, axes[0]).text(0, 1.14, f'$\\bf{{({letter})}}$ {pigment_labels[pigment]}', transform=axes[0].transAxes, va='bottom', ha='left')
radial_fig.supxlabel('Fraction of observed track', fontsize=8)
radial_fig.supylabel('Distance from eddy center (speed radii)', fontsize=8)
plt.show()
display(pigment_cells.loc[pigment_cells['variable'].eq('Tchla')].pivot(index='radial_bin', columns=['polarity', 'age_bin'], values='n_eddies'))

## Every variable on one axis

In [ ]:
final = cast(pd.DataFrame, analysis.loc[analysis['age_bin'].isin([0, N_AGE_BINS - 1])].groupby(['source', 'variable'] + identity_columns + ['age_bin'])['concentration'].mean().unstack('age_bin').dropna()).rename(columns={0: 'first', N_AGE_BINS - 1: 'last'}).reset_index()
final['change'] = final['last'] - final['first']
rng = np.random.default_rng(RANDOM_SEED)
final_rows = []
for keys, members in final.groupby(['source', 'variable', 'polarity']):
    source, variable, polarity = cast(tuple[str, str, str], keys)
    change = members['change'].to_numpy()
    initial = members['first'].to_numpy()
    draws = rng.integers(0, len(members), size=(N_BOOTSTRAP, len(members)))
    changes = change[draws].mean(axis=1)  # (n_bootstrap, n_eddies) -> (n_bootstrap,)
    initials = initial[draws].mean(axis=1)
    low, high = np.quantile(100 * changes / initials, [0.025, 0.975])
    symmetric_low, symmetric_high = np.quantile(200 * changes / (2 * initials + changes), [0.025, 0.975])
    final_rows.append({
        'source': source, 'variable': variable, 'polarity': polarity, 'n_eddies': len(members),
        'first': initial.mean(), 'last': (initial + change).mean(),
        'percent_change': 100 * change.mean() / initial.mean(), 'ci_low': low, 'ci_high': high,
        'symmetric_change': 200 * change.mean() / (2 * initial.mean() + change.mean()), 'symmetric_low': symmetric_low, 'symmetric_high': symmetric_high,
    })
final_change = pd.DataFrame(final_rows)
final_change['excludes_zero'] = (final_change['ci_low'] > 0) | (final_change['ci_high'] < 0)
variable_order = [('chl', 'CHL')] + [('plankton', group) for group in groups] + [('sdp', pigment) for pigment in pigments]
positions = {}
position = 0
for source, variable in variable_order:
    if not positions or source != variable_order[len(positions) - 1][0]:
        position += 1.5
    positions[variable] = position
    position += 1


def draw_summary_rows(ax: Axes) -> None:
    for source, label in (('chl', 'Copernicus CHL'), ('plankton', 'Copernicus plankton groups'), ('sdp', 'SDP pigments')):
        first = min(positions[variable] for member_source, variable in variable_order if member_source == source)
        ax.text(0.01, first - 1, label, transform=ax.get_yaxis_transform(), ha='left', va='center', color='#444444')
    ax.yaxis.set_ticks(list(positions.values()), [variable_labels[variable] for variable in positions])
    ax.set_ylim(position - 0.5, 0)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)


def draw_change_summary(ax: Axes, value: str, low: str, high: str) -> None:
    ax.axvline(0, color='#999999', linewidth=0.6, zorder=1)
    for polarity, color in polarity_colors.items():
        rows = final_change.loc[final_change['polarity'].eq(polarity)]
        y = rows['variable'].map(positions)
        ax.errorbar(rows[value], y, xerr=[rows[value] - rows[low], rows[high] - rows[value]], fmt='none', ecolor=color, elinewidth=0.8, capsize=1.5, capthick=0.8, zorder=2)
        for excludes_zero, face in ((True, color), (False, 'white')):
            subset = rows['excludes_zero'].eq(excludes_zero)
            ax.plot(rows.loc[subset, value], y[subset], 'o', color=color, markerfacecolor=face, markersize=4.5, markeredgewidth=0.9, zorder=3)
    draw_summary_rows(ax)
    locator = MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10])
    ticks = cast(np.ndarray, locator.tick_values(*ax.get_xlim()))
    ax.xaxis.set_ticks(ticks)
    ax.set_xlim(ticks[0], ticks[-1])


summary_fig, summary_axes = cast(tuple[Figure, np.ndarray], plt.subplots(1, 2, figsize=(6.69, 6.6), sharey=True, layout='constrained'))
change_ax, level_ax = (cast(Axes, ax) for ax in summary_axes)
draw_change_summary(change_ax, 'percent_change', 'ci_low', 'ci_high')
change_ax.set_title('$\\bf{(a)}$ Change from the first to the last fifth of the track', loc='left')
change_ax.set_xlabel('Change (% of the first fifth)')
for polarity, color in polarity_colors.items():
    rows = final_change.loc[final_change['polarity'].eq(polarity)]
    y = rows['variable'].map(positions)
    level_ax.hlines(y, rows['first'], rows['last'], color=color, linewidth=0.8, zorder=2)
    level_ax.plot(rows['first'], y, '|', color=color, markersize=6, markeredgewidth=1.2, zorder=3)
    for excludes_zero, face in ((True, color), (False, 'white')):
        subset = rows['excludes_zero'].eq(excludes_zero)
        level_ax.plot(rows.loc[subset, 'last'], y[subset], 'o', color=color, markerfacecolor=face, markersize=4.5, markeredgewidth=0.9, zorder=4)
level_ax.set(xscale='log')
level_ax.xaxis.set_ticks([0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5], ['0.002', '0.005', '0.01', '0.02', '0.05', '0.1', '0.2', '0.5'])
level_ax.xaxis.set_ticks([], minor=True)
level_ax.set_xlim(0.002, 0.6)
level_ax.tick_params(axis='y', length=0)
level_ax.grid(axis='x', color='#e5e5e5', linewidth=0.5)
level_ax.set_axisbelow(True)
level_ax.set_title('$\\bf{(b)}$ First and last fifth of the track', loc='left')
level_ax.set_xlabel('Interior concentration (mg m$^{-3}$)')
summary_fig.legend(handles=[
    Line2D([], [], color=polarity_colors[polarity], marker='o', markersize=4.5, markeredgewidth=0.9, linewidth=0.8, label=target_labels[polarity]) for polarity in polarity_names
] + [
    Line2D([], [], color='#444444', marker='o', markerfacecolor='white', markersize=4.5, markeredgewidth=0.9, linewidth=0, label='95% interval includes zero'),
    Line2D([], [], color='#444444', marker='|', markersize=6, markeredgewidth=1.2, linewidth=0, label='First fifth of the track'),
    Line2D([], [], color='#444444', marker='o', markersize=4.5, markeredgewidth=0.9, linewidth=0, label='Last fifth of the track'),
], loc='outside lower center', ncol=3)
plt.show()
display(final_change.drop(columns='excludes_zero').round(4))

In [ ]:
symmetric_fig, symmetric_ax = cast(tuple[Figure, Axes], plt.subplots(figsize=(4.2, 6.6), layout='constrained'))
draw_change_summary(symmetric_ax, 'symmetric_change', 'symmetric_low', 'symmetric_high')
symmetric_ax.set_xlabel('Symmetric change from the first to the last fifth of the track (%)')
symmetric_fig.legend(handles=[
    Line2D([], [], color=polarity_colors[polarity], marker='o', markersize=4.5, markeredgewidth=0.9, linewidth=0.8, label=target_labels[polarity]) for polarity in polarity_names
] + [Line2D([], [], color='#444444', marker='o', markerfacecolor='white', markersize=4.5, markeredgewidth=0.9, linewidth=0, label='95% interval includes zero')], loc='outside lower center', ncol=2)
plt.show()